# SORT1 locus — batch-score 5 variants

Reproduces the MCP `score_variant_batch` call. Scores N variants
across the supplied AlphaGenome tracks and ranks them by max
effect magnitude across layers.


In [ ]:
# All imports the notebook needs — top-level, no later imports.
import json
from pathlib import Path

import chorus
from chorus.analysis.normalization import get_normalizer
from chorus.analysis.batch_scoring import score_variant_batch
from chorus.analysis.analysis_request import AnalysisRequest

In [ ]:
WALKTHROUGH_DIR = Path.cwd()
print(f"Writing artifacts to: {WALKTHROUGH_DIR}")

In [ ]:
oracle = chorus.create_oracle(
    oracle_name='alphagenome',
    use_environment=True,
)
oracle.load_pretrained_model()

In [ ]:
# Inputs.
oracle_name = 'alphagenome'
gene_name = 'SORT1'
top_n = 20
variants = [
    {'chrom': 'chr1', 'pos': 109274968, 'ref': 'G', 'alt': 'T', 'id': 'rs12740374'},
    {'chrom': 'chr1', 'pos': 109275684, 'ref': 'G', 'alt': 'T', 'id': 'rs1626484'},
    {'chrom': 'chr1', 'pos': 109275216, 'ref': 'T', 'alt': 'C', 'id': 'rs660240'},
    {'chrom': 'chr1', 'pos': 109279175, 'ref': 'G', 'alt': 'A', 'id': 'rs4970836'},
    {'chrom': 'chr1', 'pos': 109274570, 'ref': 'A', 'alt': 'G', 'id': 'rs7528419'},
]
assay_ids = [
    'DNASE/EFO:0001187 DNase-seq/.',
    'CHIP_TF/EFO:0001187 TF ChIP-seq CEBPA genetically modified (insertion) using CRISPR targeting H. sapiens CEBPA/.',
    'CHIP_TF/EFO:0001187 TF ChIP-seq CEBPB/.',
    'CHIP_HISTONE/EFO:0001187 Histone ChIP-seq H3K27ac/.',
    'CAGE/hCAGE EFO:0001187/+',
    'CAGE/hCAGE EFO:0001187/-',
]

In [ ]:
# Run batch scoring. Each variant is scored multi-layer with the same
# scorer the single-variant API uses; the batch wrapper ranks results.
normalizer = get_normalizer(oracle_name=oracle_name)
analysis_request = AnalysisRequest(
    user_prompt=f"Batch-score {len(variants)} SORT1-locus variants.",
    tool_name="score_variant_batch",
    oracle_name=oracle_name,
    tracks_requested=f"{len(assay_ids)} AlphaGenome tracks",
)
batch_result = score_variant_batch(
    oracle=oracle,
    variants=variants,
    assay_ids=assay_ids,
    gene_name=gene_name,
    normalizer=normalizer,
    analysis_request=analysis_request,
    oracle_name=oracle_name,
)

In [ ]:
# Save the same artifacts the MCP tool would produce:
#   - example_output.md  (markdown report)
#   - example_output.json (structured scores)
#   - example_output.tsv (track-level table)
#   - batch_sort1_locus_scoring.html (interactive IGV report)
WALKTHROUGH_DIR.joinpath("example_output.md").write_text(batch_result.to_markdown())
WALKTHROUGH_DIR.joinpath("example_output.json").write_text(
    json.dumps(batch_result.to_dict(), indent=2, default=str)
)
try:
    batch_result.to_dataframe().to_csv(
        WALKTHROUGH_DIR / "example_output.tsv", sep="\t", index=False,
    )
except Exception as exc:
    print(f"TSV write skipped: {exc}")

_html_path = batch_result.to_html(output_path=str(WALKTHROUGH_DIR / "batch_sort1_locus_scoring.html"))
print(f"Wrote artifacts to { WALKTHROUGH_DIR }")


## What this notebook produced

- `example_output.md` — ranked variant table (top 20)
- `example_output.json` — per-variant per-layer scores
- `example_output.tsv` — flat table
- `batch_sort1_locus_scoring.html` — batch HTML report
